# **Aplicación práctica: formas anatómicas 3D con la función de Christoffel–Darboux**

**Pregunta del asesor.** ¿Sirve el método con datos reales de otro dominio? Se evalúan sus dos usos con formas anatómicas:

1. **Agrupamiento (Nivel 1): «encerrar la forma».** En la nube de puntos de un paciente, con todos sus órganos juntos y sin etiquetas, ¿se separa cada órgano? Después, ¿el conjunto de nivel del polinomio de Christoffel de cada grupo *encierra* al órgano sin invadir a los vecinos?
2. **Embedding (Nivel 2): comparar formas entre sí.** Con un descriptor por órgano, invariante a la pose, ¿quedan cerca los órganos del mismo tipo, aunque vengan de pacientes distintos?

### **Elección del dataset**

| Opción | Qué contiene | ¿Encaja? |
|---|---|---|
| **MedShapeNetCore** (Zenodo 10617451) | Formas anatómicas ya extraídas como máscara, **nube de puntos** y malla, en `.npz`; CC BY 4.0 | **Sí.** Ya son nubes de puntos y las etiquetas (órgano) sirven sólo para evaluar. Se usa el subconjunto **FLARE** (órganos abdominales: hígado, riñones, bazo, páncreas, aorta, vena cava, suprarrenales, vesícula, esófago, estómago, duodeno). |
| SELMA3D | Microscopía de hoja de luz de cerebros (TIFF de 16 bits; parches anotados en NIfTI); CC BY-NC | Parcial. Habría que segmentar volúmenes muy grandes antes de tener puntos; los parches anotados de núcleos o placas podrían servir para el Nivel 1 (grupos de células). Queda como extensión. |
| Human Organ Atlas | Volúmenes de tomografía de contraste de fase de órganos completos (resolución micrométrica) | No por ahora: volúmenes de gran tamaño sin segmentaciones listas; requeriría un paso de segmentación que no forma parte de la tesis. |

**Cómo citar los datos.** Li, J., Zhou, Z., Yang, J., *et al.* (2023). *MedShapeNet: A Large-Scale Dataset of 3D Medical Shapes for Computer Vision*. arXiv:2308.16139. DOI 10.5281/zenodo.10617451. Además, citar el dataset de origen (FLARE).

### **Cómo ejecutar**

1. Descargar `medshapenetcore_FLARE.npz` (≈ 555 MB) desde el registro de Zenodo, o con el paquete oficial: `pip install MedShapeNetCore` y `python -m MedShapeNetCore download FLARE` (lo guarda en `./medshapenetcore_npz/`). También se puede activar `DESCARGAR = True` abajo.
2. Escribir la ruta en `RUTA_NPZ` (celda 0) o dejarla en `None` para que se busque en `./medshapenetcore_npz/`, `<repo>/data/` y `~/Downloads`.
3. *Restart & Run All.* Con FLARE completo tarda ≈ 10–20 min. Otros subconjuntos de MedShapeNetCore con varias clases también funcionan cambiando `DATASET`.

Las celdas no suponen una estructura fija del `.npz`: primero la exploran y luego extraen todas las nubes que encuentran, con su clase y su índice de paciente.

In [1]:
# ===================== EDITAR AQUÍ =====================
DATASET = "FLARE"                 # subconjunto de MedShapeNetCore
RUTA_NPZ = None                   # p. ej. "~/Downloads/medshapenetcore_FLARE.npz"
DESCARGAR = False                 # True: descarga desde Zenodo si no se encuentra el archivo
N_ESCENAS = 6                     # pacientes para el Nivel 1
PUNTOS_POR_ORGANO = 400           # puntos de cada órgano dentro de la escena
GRADOS_ENVOLVENTE = (2, 4)        # grados del polinomio que "encierra" cada grupo (2: elipsoide; 4: sigue la superficie)
GRADO_FIGURA = 4                  # grado que se dibuja
PERCENTIL_ENVOLVENTE = 95         # nivel: percentil de q sobre los puntos del propio grupo
MAX_POR_CLASE = 40                # formas por órgano para el Nivel 2
N_PUNTOS = 1024                   # puntos por forma en el Nivel 2
POSE_ALEATORIA = True             # Nivel 2: quitar la posición/orientación del TAC (sólo forma)
UNIR_IZQ_DER = True               # el descriptor es invariante a reflexiones: riñón izq. y der. son la misma forma
SEMILLA = 0
GUARDAR = True                    # figuras en notebooks/figuras_aplicacion/, CSV en benchmarks/results/
# =======================================================

import os, sys, time, json, hashlib, re, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings("ignore", category=FutureWarning)

RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
for p in (RAIZ / "src", RAIZ / "benchmarks"):
    if str(p) not in sys.path: sys.path.insert(0, str(p))

from pointfield.learning import (ChristoffelDarboux, CDDescriptor, CDGlobalClustering, CDLocalSpectral, BlockScaler,
                                 MultiViewMetric, moment_descriptor, moment_block_slices, make_view, paired_bootstrap)
from pointfield.learning.descriptors import d2_histogram
from pointfield.learning.evaluation import average_precision
from pointfield.geometry.mesh import TriangleMesh
from pointfield.objects import random_similarity
from common import METHODS
from objetos_invariantes import perturb
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, homogeneity_score, completeness_score
from sklearn.decomposition import PCA
from scipy.optimize import linear_sum_assignment

FIG = RAIZ / "notebooks" / "figuras_aplicacion"; RES = RAIZ / "benchmarks" / "results"
if GUARDAR: FIG.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
_nfig = [0]
def guardar(fig, nombre):
    if not GUARDAR: return
    _nfig[0] += 1; ruta = FIG / f"{_nfig[0]:02d}_{nombre}.png"
    fig.savefig(ruta, dpi=160, bbox_inches="tight"); print(f"[figura] {ruta.relative_to(RAIZ)}")
def tabla(df, dec=3, titulo=None):
    if titulo: display(Markdown(f"**{titulo}**"))
    display(df.round(dec) if dec is not None else df)
def rng_(k=0): return np.random.default_rng(SEMILLA + k)
print("raíz del repositorio:", RAIZ)

raíz del repositorio: /home/icantu24/Documents/PEF-CD/point-field


## **1. Localizar, descargar (opcional) y explorar el archivo**

In [11]:
%pip install trimesh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.4/755.4 kB 19.8 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:

from pathlib import Path
import numpy as np
import time

# Clase necesaria para deserializar data.npy
class MyDict(dict):
    pass

ARCHIVO = Path("/home/icantu24/Downloads/data.npy")

assert ARCHIVO.is_file(), f"No existe el archivo: {ARCHIVO}"

t0 = time.time()

if ARCHIVO.suffix == ".npy":
    DATA = np.load(ARCHIVO, allow_pickle=True)
    
print(f"cargado en {time.time() - t0:.1f} s")


def describir(obj, nivel=0, max_claves=14, max_nivel=4):
    """Describe recursivamente la estructura de los datos."""
    pre = "    " * nivel

    if isinstance(obj, dict):
        claves = list(obj.keys())
        print(f"{pre}dict con {len(claves)} claves")

        if nivel >= max_nivel:
            return

        for k in claves[:max_claves]:
            print(f"{pre}  [{k!r}]")
            describir(obj[k], nivel + 1, max_claves, max_nivel)

        if len(claves) > max_claves:
            print(f"{pre}  ... ({len(claves)-max_claves} claves más)")

    elif isinstance(obj, np.ndarray):
        print(f"{pre}ndarray {obj.shape}, dtype={obj.dtype}")

        if obj.dtype == object and obj.size and nivel < max_nivel:
            if obj.ndim == 0:
                describir(obj.item(), nivel + 1, max_claves, max_nivel)
            else:
                print(f"{pre}  Primer elemento:")
                describir(obj.flat[0], nivel + 1, max_claves, max_nivel)

    elif isinstance(obj, (list, tuple)):
        print(f"{pre}{type(obj).__name__} de {len(obj)} elementos")

        if len(obj) and nivel < max_nivel:
            describir(obj[0], nivel + 1, max_claves, max_nivel)

    elif hasattr(obj, "vertices") and hasattr(obj, "faces"):
        print(f"{pre}Malla 3D ({type(obj).__name__})")
        print(f"{pre}  Vértices: {np.asarray(obj.vertices).shape}")
        print(f"{pre}  Caras: {np.asarray(obj.faces).shape}")

    else:
        print(f"{pre}{type(obj).__name__}: {str(obj)[:80]}")


# Desempaquetar el objeto principal
if isinstance(DATA, np.ndarray) and DATA.ndim == 0:
    DATA = DATA.item()

describir(DATA, max_claves=10, max_nivel=5)

cargado en 27.5 s
dict con 13 claves
  ['liver']
    dict con 3 claves
      ['mask']
        ndarray (50, 256, 256, 256), dtype=int16
      ['point']
        ndarray (50, 65536, 3), dtype=float64
      ['mesh']
        dict con 2 claves
          ['vertices']
            ndarray (50,), dtype=object
              Primer elemento:
                ndarray (54460, 3), dtype=float64
          ['faces']
            ndarray (50, 65536, 3), dtype=int16
  ['right_kidney']
    dict con 3 claves
      ['mask']
        ndarray (50, 256, 256, 256), dtype=int16
      ['point']
        ndarray (50, 65536, 3), dtype=float64
      ['mesh']
        dict con 2 claves
          ['vertices']
            ndarray (50,), dtype=object
              Primer elemento:
                ndarray (24173, 3), dtype=float64
          ['faces']
            ndarray (50,), dtype=object
              Primer elemento:
                ndarray (65536, 3), dtype=int16
  ['spleen']
    dict con 3 claves
      ['mask']
        n

## **2. Extraer las nubes de puntos con su clase y su paciente**

Se recorre la estructura y se toma, para cada forma, la nube de puntos (`point`); si no existe, se muestrea la malla (`mesh`) uniformemente en área; si sólo hay máscara (`mask`), se usan las coordenadas de sus vóxeles. La **clase** es la parte de la ruta que no es formato ni índice (en FLARE, el nombre del órgano) y el **paciente** es el índice de muestra.

In [14]:
FORMATOS = {"point", "points", "orgna", "mesh", "mask", "vertices", "faces", "label", "labels"}
def _es_indice(k): return isinstance(k, (int, np.integer)) or (isinstance(k, str) and k.isdigit())
def _numerico(a): return isinstance(a, np.ndarray) and a.dtype != object and np.issubdtype(a.dtype, np.number)

def recorrer(obj, ruta=()):
    "Genera (ruta, formato, valor) para cada nube, malla o máscara encontrada."
    if isinstance(obj, dict):
        if "vertices" in obj and "faces" in obj:
            V, F = obj["vertices"], obj["faces"]
            if _numerico(V) and V.ndim == 2:
                yield ruta, "mesh", (V, F); return
            claves = V.keys() if isinstance(V, dict) else range(len(V))
            for i in claves:
                yield ruta + (i,), "mesh", (V[i], F[i])
            return
        for k, v in obj.items():
            yield from recorrer(v, ruta + (k,))
    elif isinstance(obj, (list, tuple)) or (isinstance(obj, np.ndarray) and obj.dtype == object):
        for i, v in enumerate(obj):
            yield from recorrer(v, ruta + (i,))
    elif _numerico(obj):
        if obj.ndim == 2 and obj.shape[1] == 3 and len(obj) >= 20: yield ruta, "point", obj
        elif obj.ndim == 3 and obj.min() >= 0 and (obj > 0).sum() >= 20: yield ruta, "mask", obj

def a_nube(formato, valor, n=None, rng=None):
    if formato == "point": X = np.asarray(valor, float)
    elif formato == "mesh":
        V, F = np.asarray(valor[0], float), np.asarray(valor[1], int)
        X = TriangleMesh(V, F).sample(n or 4096, seed=int(rng.integers(2**31)) if rng is not None else 0)
    else: X = np.argwhere(np.asarray(valor) > 0).astype(float)
    X = X[np.isfinite(X).all(axis=1)]
    return np.unique(X, axis=0) if formato != "mesh" else X

PRIORIDAD = {"point": 0, "mesh": 1, "mask": 2}
formas = {}
for ruta, formato, valor in recorrer(DATA):
    partes = [str(p) for p in ruta if not (isinstance(p, str) and p.lower() in FORMATOS)]
    idx = [p for p in partes if _es_indice(p)]
    clase = "/".join(p for p in partes if not _es_indice(p)) or DATASET
    clave = (clase, idx[-1] if idx else "0")
    if clave not in formas or PRIORIDAD[formato] < PRIORIDAD[formas[clave][0]]:
        formas[clave] = (formato, valor)

FORMAS = []
for (clase, paciente), (formato, valor) in sorted(formas.items(), key=lambda t: (t[0][0], int(t[0][1]) if t[0][1].isdigit() else 0)):
    X = a_nube(formato, valor, 4096, rng_(1))
    if len(X) >= 50: FORMAS.append({"clase": clase, "paciente": paciente, "formato": formato, "X": X})
CAT = pd.DataFrame([{k: v for k, v in f.items() if k != "X"} | {"n_puntos": len(f["X"]), "extensión": float(np.ptp(f["X"], axis=0).max())}
                    for f in FORMAS])
resumen = CAT.groupby("clase").agg(formas=("paciente", "size"), formato=("formato", "first"),
                                    puntos_mediana=("n_puntos", "median"), extensión_mediana=("extensión", "median"))
tabla(resumen, 1, f"{DATASET}: {len(FORMAS)} formas en {len(resumen)} clases")
assert len(resumen) >= 2, "Se encontró una sola clase: este subconjunto no permite evaluar contra etiquetas. Probar con FLARE."

ValueError: finite vertices and valid non-negative face indices required

## **3. Galería por clase y comprobación del sistema de coordenadas**

Si las formas de un mismo paciente comparten el sistema de coordenadas del TAC (cada órgano en su lugar), la unión de los órganos de un paciente es una **escena anatómica real**. Si cada forma viene normalizada por separado, todos los órganos se encimarían y la escena se compone colocándolos en una rejilla (eso se indica claramente).

In [ ]:
clases = list(resumen.index)
cols = min(6, len(clases)); filas = int(np.ceil(len(clases) / cols))
fig = plt.figure(figsize=(2.7 * cols, 2.8 * filas))
for i, c in enumerate(clases):
    X = next(f["X"] for f in FORMAS if f["clase"] == c)
    X = X[rng_(2).choice(len(X), min(len(X), 2500), replace=False)]
    a = fig.add_subplot(filas, cols, i + 1, projection="3d"); a.scatter(*X.T, s=0.4, c=X[:, 2], cmap="viridis")
    a.set_box_aspect(np.ptp(X, axis=0) / np.ptp(X, axis=0).max()); a.set_xticks([]); a.set_yticks([]); a.set_zticks([])
    a.set_title(c, fontsize=9)
fig.suptitle(f"{DATASET}: un ejemplo por clase"); plt.tight_layout(); guardar(fig, "galeria_clases"); plt.show()

# ¿Comparten coordenadas los órganos de un paciente?
por_paciente = CAT.groupby("paciente").clase.nunique().sort_values(ascending=False)
razones = []
for p in por_paciente.index[:20]:
    F = [f["X"] for f in FORMAS if f["paciente"] == p]
    if len(F) < 3: continue
    C = np.array([X.mean(0) for X in F]); r = np.median([np.sqrt(((X - X.mean(0)) ** 2).sum(1).mean()) for X in F])
    d = np.sqrt(((C[:, None] - C[None]) ** 2).sum(-1))[np.triu_indices(len(C), 1)]
    razones.append(np.median(d) / r)
ALINEADO = bool(razones) and np.median(razones) > 1.0
print(f"pacientes con ≥ 3 clases: {int((por_paciente >= 3).sum())}; distancia mediana entre centroides / radio mediano = "
      f"{np.median(razones) if razones else float('nan'):.2f}")
display(Markdown("**Las formas comparten el sistema de coordenadas del paciente: las escenas del Nivel 1 son anatómicas reales.**" if ALINEADO else
                 "**Las formas no comparten coordenadas (o hay pocos pacientes con varias clases): las escenas del Nivel 1 se componen en una rejilla. "
                 "El resultado mide si se separan formas distintas, no órganos en contacto.**"))

---
## **4. Nivel 1 — agrupar los órganos de un paciente y «encerrar» cada forma**

Para cada paciente se juntan todos sus órganos en una sola nube (cada órgano aporta `PUNTOS_POR_ORGANO` puntos), se normaliza la escena (centro y radio RMS) y se agrupa **sin etiquetas** con los mismos métodos y configuraciones de la tesis (§5.4), con $K$ = número de órganos del paciente. Las etiquetas sólo se usan al final para ARI, NMI, homogeneidad y completitud.

In [ ]:
METODOS_N1 = {k: v for k, v in METHODS.items() if "Szegő" not in k}   # la fase V2 sólo está definida en 2D
NOMBRE_CORTO = {"CD global (inicial)": "V0 · CD global", "CD local-espectral (mejora 1)": "V1 · CD local", "Spectral (kNN)": "Espectral k-NN"}

def escena(paciente, rng):
    F = [f for f in FORMAS if f["paciente"] == paciente]
    Xs, ys, nombres = [], [], []
    for j, f in enumerate(F):
        X = f["X"][rng.choice(len(f["X"]), min(PUNTOS_POR_ORGANO, len(f["X"])), replace=False)]
        if not ALINEADO:   # composición en rejilla: cada forma normalizada y desplazada
            X = (X - X.mean(0)) / np.sqrt(((X - X.mean(0)) ** 2).sum(1).mean()) + 3.0 * np.array([j % 4, j // 4, 0.0])
        Xs.append(X); ys.append(np.full(len(X), j)); nombres.append(f["clase"])
    X = np.vstack(Xs); X = (X - X.mean(0)) / np.sqrt(((X - X.mean(0)) ** 2).sum(1).mean())
    return X, np.concatenate(ys), nombres

PACIENTES = [p for p in por_paciente.index if por_paciente[p] >= 3][:N_ESCENAS] or list(por_paciente.index[:N_ESCENAS])
filas, ESCENAS = [], {}
for p in PACIENTES:
    X, y, nombres = escena(p, rng_(3)); K = len(nombres); ESCENAS[p] = {"X": X, "y": y, "nombres": nombres, "pred": {}}
    for m, (fabrica, _, _, _) in METODOS_N1.items():
        t0 = time.perf_counter(); lab = np.asarray(fabrica(K).fit_predict(X)); dt = time.perf_counter() - t0
        ESCENAS[p]["pred"][m] = lab
        filas.append({"paciente": p, "órganos": K, "puntos": len(X), "método": NOMBRE_CORTO.get(m, m), "ARI": adjusted_rand_score(y, lab),
                      "NMI": normalized_mutual_info_score(y, lab), "homogeneidad": homogeneity_score(y, lab),
                      "completitud": completeness_score(y, lab), "tiempo (s)": dt})
N1 = pd.DataFrame(filas)
tabla(N1.pivot_table(index="paciente", columns="método", values="ARI"), 2, "Nivel 1 · ARI por paciente")
tabla(N1.groupby("método")[["ARI", "NMI", "homogeneidad", "completitud", "tiempo (s)"]].agg(["mean", "std"]).sort_values(("ARI", "mean"), ascending=False),
      3, f"Nivel 1 · promedio sobre {len(PACIENTES)} pacientes")

In [ ]:
p0 = PACIENTES[0]; E = ESCENAS[p0]
mostrar = ["verdad", "CD local-espectral (mejora 1)", "Spectral (kNN)", "CD global (inicial)", "K-Means", "DBSCAN"]
fig = plt.figure(figsize=(18, 6))
for i, m in enumerate(mostrar):
    lab = E["y"] if m == "verdad" else E["pred"][m]
    a = fig.add_subplot(2, 3, i + 1, projection="3d"); a.scatter(*E["X"].T, color=plt.get_cmap("tab20")(lab % 20), s=1.5)
    t = "Etiquetas reales (sólo evaluación)" if m == "verdad" else f"{NOMBRE_CORTO.get(m, m)} · ARI = {adjusted_rand_score(E['y'], lab):.2f}"
    a.set_title(t, fontsize=10); a.set_xticks([]); a.set_yticks([]); a.set_zticks([])
fig.suptitle(f"Paciente {p0}: {len(E['nombres'])} órganos ({', '.join(E['nombres'])})", fontsize=10)
plt.tight_layout(); guardar(fig, "nivel1_escena"); plt.show()

### **4.1 «Encerrar la forma»: envolvente de Christoffel de cada grupo**

Para cada grupo encontrado por V1 se ajusta un polinomio de Christoffel de grado 2 y de grado 4 (`GRADOS_ENVOLVENTE`) (normalizado por grupo, regularización $c = 10^{-6}$) y se define su envolvente como el conjunto de subnivel $\{x : q(x) \le \gamma\}$, con $\gamma$ el percentil `PERCENTIL_ENVOLVENTE` de $q$ sobre los puntos del propio grupo (§2.6.4). Se mide:

- **cobertura**: fracción de los puntos del órgano real asociado (asignación húngara) que quedan dentro de la envolvente;
- **invasión**: fracción de los puntos *de otros órganos* que quedan dentro de la envolvente;
- **pureza**: de los puntos que la envolvente encierra, fracción que pertenece a su órgano.

La misma medición con los grupos reales en lugar de los de V1 da la referencia de qué tan bien puede encerrar el polinomio a cada órgano. Con grado 2 la envolvente es un elipsoide que encierra todo el órgano (incluido su interior, si los puntos son de superficie); con grado 4 se ciñe a la forma, como se vio en §6.9.

In [ ]:
def envolventes(X, lab, grado):
    "Ajusta un modelo CD por grupo; devuelve el modelo, la normalización y el umbral."
    out = {}
    for g in np.unique(lab):
        if g < 0 or (lab == g).sum() < 50: continue
        Z = X[lab == g]; c = Z.mean(0); r = np.sqrt(((Z - c) ** 2).sum(1).mean())
        cd = ChristoffelDarboux(degree=grado, reg=1e-6).fit((Z - c) / r)
        out[g] = (cd, c, r, np.percentile(cd.score((Z - c) / r), PERCENTIL_ENVOLVENTE))
    return out
def dentro(env, X):
    return {g: cd.score((X - c) / r) <= gam for g, (cd, c, r, gam) in env.items()}

def medir_envolventes(X, y, lab, grado):
    env = envolventes(X, lab, grado); D = dentro(env, X)
    gs = list(env); ys = np.unique(y)
    C = np.array([[np.mean(lab[y == t] == g) for g in gs] for t in ys]); fi, co = linear_sum_assignment(-C)
    filas = []
    for t, j in zip(ys[fi], co):
        g = gs[j]; ins = D[g]
        filas.append({"órgano": t, "cobertura": ins[y == t].mean(), "invasión": ins[y != t].mean(),
                      "pureza": (y[ins] == t).mean() if ins.any() else np.nan})
    return pd.DataFrame(filas), env

filas = []
for p, E in ESCENAS.items():
    for grado in GRADOS_ENVOLVENTE:
        for fuente, lab in (("grupos reales", E["y"]), ("grupos de V1", E["pred"]["CD local-espectral (mejora 1)"])):
            T, _ = medir_envolventes(E["X"], E["y"], lab, grado); T["paciente"] = p; T["grupos"] = fuente; T["grado"] = grado
            T["órgano"] = [E["nombres"][t] for t in T["órgano"]]; filas.append(T)
ENV = pd.concat(filas)
tabla(ENV[ENV.grado == GRADO_FIGURA].groupby(["grupos", "órgano"])[["cobertura", "invasión", "pureza"]].mean().unstack(0), 3,
      f"Envolventes de grado {GRADO_FIGURA} por órgano (promedio sobre pacientes)")
tabla(ENV.groupby(["grado", "grupos"])[["cobertura", "invasión", "pureza"]].agg(["mean", "std"]), 3, "Envolventes: resumen por grado")

In [ ]:
from skimage.measure import marching_cubes
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
E = ESCENAS[p0]; lab = E["pred"]["CD local-espectral (mejora 1)"]; env = envolventes(E["X"], lab, GRADO_FIGURA)
lo, hi = E["X"].min(0) - 0.15, E["X"].max(0) + 0.15; n = 72
g1 = [np.linspace(lo[i], hi[i], n) for i in range(3)]; G = np.stack(np.meshgrid(*g1, indexing="ij"), -1).reshape(-1, 3)
fig = plt.figure(figsize=(15, 6)); a = fig.add_subplot(1, 2, 1, projection="3d"); b = fig.add_subplot(1, 2, 2)
cmap = plt.get_cmap("tab20")
for g, (cd, c, r, gam) in env.items():
    S = np.log10(cd.score((G - c) / r)).reshape(n, n, n)
    try:
        V, Fc, _, _ = marching_cubes(S, np.log10(gam), spacing=tuple((hi - lo) / (n - 1)))
        a.add_collection3d(Poly3DCollection(V[Fc] + lo, alpha=0.25, facecolor=cmap(g % 20), edgecolor="none"))
    except (ValueError, RuntimeError):
        pass
    k = n // 2; b.contour(g1[0], g1[1], S[:, :, k].T, levels=[np.log10(gam)], colors=[cmap(g % 20)])
a.scatter(*E["X"].T, color=cmap(lab % 20), s=0.3, alpha=0.4); a.set_xlim(lo[0], hi[0]); a.set_ylim(lo[1], hi[1]); a.set_zlim(lo[2], hi[2])
a.set_title(f"Envolventes de Christoffel (grado {GRADO_FIGURA}) de los grupos de V1"); a.set_xticks([]); a.set_yticks([]); a.set_zticks([])
zc = g1[2][n // 2]; m_ = np.abs(E["X"][:, 2] - zc) < 0.08
b.scatter(E["X"][m_, 0], E["X"][m_, 1], color=cmap(lab[m_] % 20), s=4); b.set_aspect("equal")
b.set_title(f"Corte z = {zc:.2f}: puntos cercanos al corte y contornos q = γ")
plt.tight_layout(); guardar(fig, "nivel1_envolventes"); plt.show()

---
## **5. Nivel 2 — embedding de formas: ¿se acercan los órganos del mismo tipo?**

Cada forma se describe con 1 024 puntos. Con `POSE_ALEATORIA = True` se le aplica una similitud aleatoria (rotación, reflexión, escala y traslación), para que la comparación dependa sólo de la forma y no de dónde estaba el órgano en el TAC. Se comparan los descriptores de la tesis: `CD_U` (Christoffel–Darboux), `M_U` (momentos, control) y D2 (distancias entre pares), cada uno sin y con la métrica multivista aprendida **sin etiquetas** (§4.4).

Como el descriptor es invariante a reflexiones, los órganos pares (riñón y suprarrenal izquierdos y derechos) son la misma forma especular; con `UNIR_IZQ_DER = True` se evalúan como una sola clase.

In [ ]:
def clase_n2(c):
    if not UNIR_IZQ_DER: return c
    return re.sub(r"[_\s-]*(left|right|izquierd[oa]|derech[oa]|_l$|_r$|\bl\b|\br\b)", "", c, flags=re.I).strip("_ -") or c

r2 = rng_(5); SEL = []
for c in clases:
    F = [f for f in FORMAS if f["clase"] == c]
    for i in r2.permutation(len(F))[:MAX_POR_CLASE]: SEL.append(F[i])
NUBES, Y_TXT = [], []
for f in SEL:
    X = f["X"][r2.choice(len(f["X"]), N_PUNTOS, replace=len(f["X"]) < N_PUNTOS)]
    NUBES.append(random_similarity(X, r2) if POSE_ALEATORIA else X); Y_TXT.append(clase_n2(f["clase"]))
CLASES2 = sorted(set(Y_TXT)); Y2 = np.array([CLASES2.index(c) for c in Y_TXT])
tabla(pd.Series(Y_TXT).value_counts().rename("formas").to_frame(), 0, f"Nivel 2: {len(NUBES)} formas, {len(CLASES2)} clases")

t0 = time.time()
cd = CDDescriptor()
F_ = {"CD_U": np.array([cd(x) for x in NUBES]), "M_U": np.array([moment_descriptor(x) for x in NUBES]),
      "D2": np.array([d2_histogram(x) for x in NUBES])}
sc = {"CD_U": BlockScaler(cd.block_slices(3)).fit(F_["CD_U"]), "M_U": BlockScaler(moment_block_slices(3)).fit(F_["M_U"])}
vistas = [[make_view(x, r2) for _ in range(4)] for x in NUBES]
VV = {"CD_U": np.array([[sc["CD_U"].transform(cd(v)) for v in vs] for vs in vistas]),
      "M_U": np.array([[sc["M_U"].transform(moment_descriptor(v)) for v in vs] for vs in vistas])}
MET = {k: MultiViewMetric(seed=SEMILLA).fit(VV[k]) for k in VV}
EMB = {"CD_U": sc["CD_U"].transform(F_["CD_U"]), "M_U": sc["M_U"].transform(F_["M_U"]), "D2": F_["D2"]}
for k in MET: EMB[f"{k}+m"] = MET[k].transform(EMB[k])
print(f"descriptores y métricas en {time.time() - t0:.0f} s; η elegido sin etiquetas:", {k: m.eta_ for k, m in MET.items()})

In [ ]:
def ap_loo(F, y):
    "AP de cada forma consultando contra todas las demás (se excluye a sí misma)."
    D = np.sqrt(((F[:, None] - F[None]) ** 2).sum(-1)); np.fill_diagonal(D, np.inf)
    ap = np.array([average_precision(D[i][np.arange(len(y)) != i], (y == y[i])[np.arange(len(y)) != i]) for i in range(len(y))])
    nn = (y[np.argmin(D, axis=1)] == y).mean()
    return ap, nn, D

AP, filas = {}, []
for m, F in EMB.items():
    ap, nn, D = ap_loo(F, Y2); AP[m] = ap
    intra = np.mean([D[i, (Y2 == Y2[i]) & (np.arange(len(Y2)) != i)].mean() for i in range(len(Y2))])
    inter = np.mean([D[i, Y2 != Y2[i]].mean() for i in range(len(Y2))])
    filas.append({"descriptor": m, "mAP": np.nanmean(ap), "1-NN": nn, "distancia intra / inter": intra / inter})
N2 = pd.DataFrame(filas).set_index("descriptor").sort_values("mAP", ascending=False)
tabla(N2, 3, f"Nivel 2 · recuperación dejando una fuera (azar ≈ 1/{len(CLASES2)})")
filas = []
for a, b in (("CD_U+m", "M_U+m"), ("CD_U+m", "D2"), ("CD_U", "M_U")):
    bs = paired_bootstrap(AP[a], AP[b], strata=Y2, n_boot=3000, seed=SEMILLA)
    filas.append({"contraste": f"{a} − {b}", "Δ mAP": bs["diff"], "IC bajo": bs["ci_low"], "IC alto": bs["ci_high"],
                  "lectura": "CD mejor" if bs["ci_low"] > 0 else ("CD peor" if bs["ci_high"] < 0 else "sin diferencia")})
CONTRASTES2 = pd.DataFrame(filas).set_index("contraste"); tabla(CONTRASTES2, 3, "Contrastes con IC bootstrap al 95 % (estratificado por clase)")
tabla(pd.DataFrame({m: pd.Series(AP[m]).groupby(np.array(CLASES2)[Y2]).mean() for m in ("D2", "M_U+m", "CD_U+m")}), 3, "AP medio por clase")

In [ ]:
import umap
fig, ax = plt.subplots(1, 3, figsize=(18, 5.5))
cm = plt.get_cmap("tab20")
for a, (m, metodo) in zip(ax, (("CD_U+m", "UMAP"), ("CD_U+m", "PCA"), ("M_U+m", "UMAP"))):
    Z = (umap.UMAP(n_neighbors=10, min_dist=0.1, random_state=SEMILLA).fit_transform(EMB[m]) if metodo == "UMAP"
         else PCA(2).fit_transform(EMB[m]))
    for k, c in enumerate(CLASES2):
        a.scatter(*Z[Y2 == k].T, s=16, color=cm(k % 20), label=c)
    a.set_title(f"{m} · {metodo}"); a.set_xticks([]); a.set_yticks([])
ax[0].legend(fontsize=7, ncol=2, loc="best")
plt.tight_layout(); guardar(fig, "nivel2_embedding"); plt.show()

# Distancias entre centroides de clase en el embedding CD_U+m: ¿qué órganos se parecen?
F = EMB["CD_U+m"]; Cc = np.array([F[Y2 == k].mean(0) for k in range(len(CLASES2))])
Dc = np.sqrt(((Cc[:, None] - Cc[None]) ** 2).sum(-1)); disp = np.array([np.sqrt(((F[Y2 == k] - Cc[k]) ** 2).sum(1)).mean() for k in range(len(CLASES2))])
fig, ax = plt.subplots(figsize=(0.55 * len(CLASES2) + 3, 0.5 * len(CLASES2) + 2))
im = ax.imshow(Dc / disp[:, None], cmap="viridis"); ax.set_xticks(range(len(CLASES2))); ax.set_yticks(range(len(CLASES2)))
ax.set_xticklabels(CLASES2, rotation=60, ha="right", fontsize=8); ax.set_yticklabels(CLASES2, fontsize=8)
for i in range(len(CLASES2)):
    for j in range(len(CLASES2)): ax.text(j, i, f"{Dc[i, j] / disp[i]:.1f}", ha="center", va="center", fontsize=6, color="w")
plt.colorbar(im, ax=ax, label="distancia entre centroides / dispersión de la clase (fila)")
ax.set_title("CD_U+m: separación entre clases (> 1: la clase está más lejos que su propia dispersión)"); ax.grid(False)
plt.tight_layout(); guardar(fig, "nivel2_distancias_clases"); plt.show()

In [ ]:
# Vecinos más cercanos de algunas formas (CD_U+m)
_, _, D = ap_loo(EMB["CD_U+m"], Y2)
consultas = [int(np.flatnonzero(Y2 == k)[0]) for k in range(0, len(CLASES2), max(1, len(CLASES2) // 4))][:4]
fig = plt.figure(figsize=(15, 3.2 * len(consultas)))
for i, q in enumerate(consultas):
    vec = np.argsort(D[q])[:4]
    for j, o in enumerate([q, *vec]):
        a = fig.add_subplot(len(consultas), 5, 5 * i + j + 1, projection="3d"); X = NUBES[o]
        a.scatter(*X.T, s=0.6, c=X[:, 2], cmap="viridis"); a.set_xticks([]); a.set_yticks([]); a.set_zticks([])
        ok = "" if j == 0 else (" ✔" if Y2[o] == Y2[q] else " ✘")
        a.set_title(("CONSULTA: " if j == 0 else f"vecino {j}: ") + CLASES2[Y2[o]] + ok, fontsize=8)
plt.tight_layout(); guardar(fig, "nivel2_vecinos"); plt.show()

In [ ]:
# Agrupamiento de las formas sin etiquetas (K = número de clases) y matriz de confusión
K2 = len(CLASES2); filas = []
for m in ("D2", "M_U+m", "CD_U+m"):
    for nombre, mod in (("K-Means", KMeans(K2, n_init=10, random_state=SEMILLA)), ("Ward", AgglomerativeClustering(K2)),
                        ("Espectral k-NN", SpectralClustering(K2, affinity="nearest_neighbors", n_neighbors=10, random_state=SEMILLA))):
        lab = mod.fit_predict(EMB[m])
        filas.append({"descriptor": m, "agrupamiento": nombre, "ARI": adjusted_rand_score(Y2, lab), "NMI": normalized_mutual_info_score(Y2, lab)})
AGR2 = pd.DataFrame(filas).set_index(["descriptor", "agrupamiento"]); tabla(AGR2, 3, "Nivel 2 · agrupamiento de formas")

lab = SpectralClustering(K2, affinity="nearest_neighbors", n_neighbors=10, random_state=SEMILLA).fit_predict(EMB["CD_U+m"])
C = pd.crosstab(Y2, lab).reindex(index=range(K2), columns=range(K2), fill_value=0).values
fi, co = linear_sum_assignment(-C); C = C[:, co[np.argsort(fi)]]
fig, ax = plt.subplots(figsize=(0.5 * K2 + 3, 0.45 * K2 + 2.5))
ax.imshow(C / np.maximum(C.sum(1, keepdims=True), 1), cmap="Blues", vmin=0, vmax=1)
for i in range(K2):
    for j in range(K2):
        if C[i, j]: ax.text(j, i, C[i, j], ha="center", va="center", fontsize=7)
ax.set_yticks(range(K2)); ax.set_yticklabels(CLASES2, fontsize=8); ax.set_xticks(range(K2)); ax.set_xticklabels(CLASES2, rotation=60, ha="right", fontsize=8)
ax.set_xlabel("grupo encontrado (asignado a la clase con el algoritmo húngaro)"); ax.set_ylabel("clase real"); ax.grid(False)
ax.set_title("CD_U+m · espectral k-NN: matriz de confusión"); plt.tight_layout(); guardar(fig, "nivel2_confusion"); plt.show()

### **5.1 Robustez: formas incompletas o ruidosas contra una galería limpia**

Cada forma se consulta perturbada (mismas perturbaciones de §5.3) contra todas las demás formas limpias.

In [ ]:
r3 = rng_(7); filas = []
for cond in ("clean", "noise", "density", "occlusion", "outliers"):
    Q = [perturb(x, cond, r3) for x in NUBES]
    FQ = {"CD_U": sc["CD_U"].transform(np.array([cd(x) for x in Q])), "M_U": sc["M_U"].transform(np.array([moment_descriptor(x) for x in Q])),
          "D2": np.array([d2_histogram(x) for x in Q])}
    FQ["CD_U+m"] = MET["CD_U"].transform(FQ["CD_U"]); FQ["M_U+m"] = MET["M_U"].transform(FQ["M_U"])
    for m in ("D2", "M_U+m", "CD_U+m"):
        D = np.sqrt(((FQ[m][:, None] - EMB[m][None]) ** 2).sum(-1)); np.fill_diagonal(D, np.inf)
        idx = np.arange(len(Y2))
        ap = np.array([average_precision(D[i][idx != i], (Y2 == Y2[i])[idx != i]) for i in idx])
        filas.append({"condición": cond, "descriptor": m, "mAP": np.nanmean(ap)})
ROB = pd.DataFrame(filas).pivot(index="descriptor", columns="condición", values="mAP")[["clean", "noise", "density", "occlusion", "outliers"]]
tabla(ROB, 3, "Nivel 2 · mAP bajo perturbaciones de la consulta")

---
## **6. Resumen y exportación**

In [ ]:
mej = N1.groupby("método").ARI.mean().sort_values(ascending=False)
_E = ENV[ENV.grado == GRADO_FIGURA]
env_v1 = _E[_E.grupos == "grupos de V1"][["cobertura", "invasión", "pureza"]].mean()
env_gt = _E[_E.grupos == "grupos reales"][["cobertura", "invasión", "pureza"]].mean()
c = CONTRASTES2.loc["CD_U+m − M_U+m"]
texto = f'''**Resultados generados con {DATASET}** ({len(FORMAS)} formas, {len(resumen)} clases; escenas {"anatómicas reales" if ALINEADO else "compuestas en rejilla"}).

- **Nivel 1.** ARI medio sobre {len(PACIENTES)} pacientes: {", ".join(f"{k} {v:.2f}" for k, v in mej.items())}.
- **Envolventes (grado {GRADO_FIGURA}, percentil {PERCENTIL_ENVOLVENTE}).** Con los grupos de V1: cobertura {env_v1.cobertura:.2f}, invasión {env_v1['invasión']:.3f}, pureza {env_v1.pureza:.2f}. Con los grupos reales: cobertura {env_gt.cobertura:.2f}, invasión {env_gt['invasión']:.3f}, pureza {env_gt.pureza:.2f}.
- **Nivel 2.** mAP: {", ".join(f"{k} {v:.3f}" for k, v in N2.mAP.items())} (azar ≈ {1 / len(CLASES2):.2f}). CD_U+m − M_U+m = {c['Δ mAP']:+.3f} [{c['IC bajo']:+.3f}, {c['IC alto']:+.3f}] → {c.lectura}.
- **Agrupamiento de formas.** Mejor ARI: {AGR2.ARI.idxmax()[0]} con {AGR2.ARI.idxmax()[1]} ({AGR2.ARI.max():.2f}).
'''
display(Markdown(texto))
if GUARDAR:
    N1.to_csv(RES / f"aplicacion_{DATASET}_nivel1.csv", index=False)
    ENV.to_csv(RES / f"aplicacion_{DATASET}_envolventes.csv", index=False)
    N2.to_csv(RES / f"aplicacion_{DATASET}_nivel2.csv")
    CONTRASTES2.to_csv(RES / f"aplicacion_{DATASET}_contrastes.csv")
    AGR2.to_csv(RES / f"aplicacion_{DATASET}_agrupamiento_formas.csv")
    ROB.to_csv(RES / f"aplicacion_{DATASET}_robustez.csv")
    (RES / f"aplicacion_{DATASET}_resumen.md").write_text(texto, encoding="utf-8")
    print("CSV y resumen guardados en", RES.relative_to(RAIZ))

### **Cómo llevarlo a la tesis**

Sección sugerida: **6.10 Aplicación práctica: formas anatómicas (MedShapeNetCore · FLARE)**, con
(i) la galería por clase, (ii) la tabla de ARI del Nivel 1 y la figura de la escena, (iii) la tabla de envolventes y su figura,
(iv) la tabla de mAP/1-NN con los contrastes, (v) el embedding UMAP y la matriz de distancias entre clases, (vi) la tabla de robustez.

En la discusión conviene señalar: que los órganos pares son indistinguibles para un descriptor invariante a reflexiones (por diseño); que los órganos en contacto (hígado–vesícula, páncreas–duodeno) son el caso difícil del Nivel 1; y que la envolvente de grado alto encierra mejor la forma pero a costa del condicionamiento (§2.7).